# Misinformation Homophily Agent Based Model

**Papers:** list of papers

---

## 1. Background

The paper asks: Does ingroup bias slow down the spread of misinformation across a population and at what bias level does it stay confined to a single group?


## 2. Imports & Setup

In [48]:
import mesa
from mesa.space import NetworkGrid
from mesa.datacollection import DataCollector
import networkx as nx
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"Mesa version: {mesa.__version__}")

Mesa version: 3.5.1


# 3. Agent

For the agent we used a spreading model called SIS where we have two groups of individuals: susceptible individuals & infectious individuals.(Liu et al., 2019)

Infected agents recover with probability recovery_prob per step, returning to susceptible which reflects that individuals can forget or stop spreading misinformation (Liu et al., 2019).

In [49]:
class HomophilyAgent(mesa.Agent):
    def __init__(self, model: MisinformationModel, group: int):
        super().__init__(model)
        self.group = group # 0 or 1
        self.status = "S" # mark as susceptible individual

    def step(self):
        if self.status != "I":
            return

        neighbors = self.model.grid.get_neighbors(self.pos, include_center=False)
        for neighbor in neighbors:
            if neighbor.status == "S":
                p = self.model.base_prob
                # it is less likely for other groups to get infected
                if neighbor.group != self.group:
                    p *= (1 - self.model.bias)
                # randomly infect neighbors
                if self.random.random() < p:
                    neighbor.status = "I"

        # recover (SIS: back to susceptible)
        if self.random.random() < self.model.recovery_prob:
            self.status = "S"

# 4. Misinformation Model

There are two types for network_type: "random" (Erdős–Rényi) and "small_world" (Watts-Strogatz)<br>
Random = all connections share the same probability<br>
Small-World = Friends which form groups and know each other

In [50]:
class MisinformationModel(mesa.Model):
    def __init__(self, n=200, num_groups=2, bias=0.5, base_prob=0.1, recovery_prob=0.05, network_type="random", seed=None):
        super().__init__(seed=seed)

        self.n = n
        self.num_groups = num_groups
        self.bias = bias
        self.base_prob = base_prob
        self.recovery_prob = recovery_prob
        self.datacollector = DataCollector(
            model_reporters={
                "Susceptible":  lambda m: sum(1 for a in m.agents if a.status == "S") / m.n,            # percentage of susceptible agents
                "Infected":     lambda m: sum(1 for a in m.agents if a.status == "I") / m.n,            # percentage of infected agents
                "Infected_G0":  lambda m: sum(1 for a in m.agents if a.status == "I" and a.group == 0), # number of infected in G0
                "Infected_G1":  lambda m: sum(1 for a in m.agents if a.status == "I" and a.group == 1), # number of infected in G1
            }
        )
        self.datacollector.collect(self)

        if network_type == "random":
            G = nx.erdos_renyi_graph(n, 0.06, seed=seed)
        elif network_type == "small_world":
            G = nx.watts_strogatz_graph(n, k=6, p=0.1, seed=seed)
        else:
            raise ValueError(f"Unknown network_type: {network_type}")

        self.grid = NetworkGrid(G)

        # group agents
        for i, node in enumerate(G.nodes()):
            group = i % num_groups # 0 or 1
            agent = HomophilyAgent(self, group)
            self.grid.place_agent(agent, node)

        # initial agent infection
        # misinformation originates in group0
        group0 = [a for a in self.agents if a.group == 0]
        n_initial = max(1, int(len(group0) * 0.05)) # infect 5% of group size
        for agent in self.random.sample(group0, n_initial):
            agent.status = "I" # mark as infected individual

    def step(self):
         # randomly call step for an agent
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)

# 5. Plot

In [51]:
model = MisinformationModel(n=200, bias=0.9, base_prob=0.1, network_type="random", seed=42)

for _ in range(100):
    model.step()

data = model.datacollector.get_model_vars_dataframe()
print(data)

     Susceptible  Infected  Infected_G0  Infected_G1
0          0.000     0.000            0            0
1          0.955     0.045            9            0
2          0.940     0.060           12            0
3          0.915     0.085           16            1
4          0.820     0.180           31            5
..           ...       ...          ...          ...
96         0.110     0.890           86           92
97         0.095     0.905           89           92
98         0.075     0.925           94           91
99         0.085     0.915           92           91
100        0.100     0.900           91           89

[101 rows x 4 columns]


# 6. Summary